# Feature Store Architecture

Feature stores are the central nervous system of a mature ML platform — a unified layer that manages feature computation, storage, versioning, and serving for both model training and real-time inference.

**Why feature stores matter:**
- **Training-serving skew** is the #1 cause of production ML failures. A feature store guarantees the exact same computation logic is used offline (training) and online (inference).
- **Feature reuse** across teams eliminates duplicated ETL pipelines — a feature computed once can power dozens of models.
- **Point-in-time correctness** prevents data leakage: when creating a training dataset, each row is joined to feature values *as they existed at the label timestamp*, not current values.

**Architecture layers:**
| Layer | Technology | Latency | Use |
|---|---|---|---|
| Offline Store | S3 / GCS / Parquet | Minutes | Training, batch inference |
| Online Store | Redis / DynamoDB | <10 ms | Real-time inference |
| Feature Pipeline | Spark / dbt / Flink | Minutes–hours | Computation & materialization |
| Feature Registry | Feast / Tecton | N/A | Definitions, lineage, discovery |

**Topics covered:**
1. Feature store concepts and architecture
2. Defining features with Feast `FeatureView`
3. Offline vs online stores — read and write patterns
4. Point-in-time correct feature retrieval
5. Training dataset creation and serving integration
6. Best practices and anti-patterns

In [ ]:
# Cell 2: Imports, Setup, and Mock Data Generation
# pip install feast pandas numpy scikit-learn pyarrow

import os
import json
import shutil
import tempfile
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Create a temporary workspace for the Feast feature store ──
FEAST_REPO = Path(tempfile.mkdtemp()) / 'feature_repo'
FEAST_REPO.mkdir(parents=True, exist_ok=True)
DATA_DIR = FEAST_REPO / 'data'
DATA_DIR.mkdir(exist_ok=True)
print(f'Feast repo: {FEAST_REPO}')

# ── Generate synthetic e-commerce user data ──
N_USERS    = 2_000
N_EVENTS   = 20_000

user_ids = np.arange(1, N_USERS + 1)

# User profile features (slowly changing)
user_profiles = pd.DataFrame({
    'user_id':             user_ids,
    'age':                 np.random.randint(18, 70, N_USERS),
    'account_age_days':    np.random.randint(1, 1825, N_USERS),
    'country':             np.random.choice(['US', 'UK', 'DE', 'IN', 'BR'], N_USERS),
    'subscription_tier':   np.random.choice(['free', 'basic', 'premium'], N_USERS, p=[0.5, 0.35, 0.15]),
    'event_timestamp':     [datetime(2024, 1, 1)] * N_USERS,
    'created':             [datetime(2024, 1, 1)] * N_USERS,
})

# User behavioral features (frequently updated)
timestamps = [datetime(2024, 1, 1) + timedelta(hours=i) for i in
              np.random.randint(0, 24 * 365, N_EVENTS)]
user_stats = pd.DataFrame({
    'user_id':                   np.random.choice(user_ids, N_EVENTS),
    'purchases_last_30d':        np.random.poisson(3, N_EVENTS),
    'avg_order_value':           np.random.lognormal(3.5, 0.8, N_EVENTS).round(2),
    'session_count_last_7d':     np.random.poisson(5, N_EVENTS),
    'cart_abandonment_rate':     np.random.beta(2, 5, N_EVENTS).round(4),
    'days_since_last_purchase':  np.random.randint(0, 180, N_EVENTS),
    'event_timestamp':           timestamps,
    'created':                   timestamps,
})

# Save to parquet (the offline store data source)
user_profiles.to_parquet(DATA_DIR / 'user_profiles.parquet', index=False)
user_stats.to_parquet(DATA_DIR / 'user_stats.parquet', index=False)

print(f'user_profiles: {user_profiles.shape} | user_stats: {user_stats.shape}')
print('\nProfile sample:')
print(user_profiles.head(3).to_string(index=False))
print('\nStats sample:')
print(user_stats.head(3).to_string(index=False))

## Feature Store Core Concepts

### Entities
An **entity** is the primary key that features are associated with — typically a business object like `user_id`, `item_id`, or `driver_id`. Entities define the join key used when retrieving features for a prediction.

### Feature Views
A **FeatureView** is the core unit of feature definition. It specifies:
- Which **entity** it belongs to
- Which **data source** provides the raw values
- The **schema** (column names and types)
- A **TTL** (time-to-live) — how stale a feature value can be before it must be refreshed

### Data Sources
Feast supports pluggable data sources. For offline storage, you typically use **FileSource** (Parquet on local disk or S3) or **BigQuerySource**. For online storage, you configure **Redis** or **DynamoDB** as the online store backend.

### Materialization
**Materialization** is the process of reading features from the offline store and writing the latest values to the online store so they can be served at low latency. You materialize on a schedule (e.g., every hour) to keep the online store fresh.

```
Offline Store (Parquet/S3)
        │
        │  feast materialize (batch job, runs hourly)
        ▼
Online Store (Redis/DynamoDB)
        │
        │  get_online_features() → <10ms
        ▼
  ML Model Serving
```

### Point-in-Time Joins
When building a training dataset from historical labels, Feast performs a **point-in-time join**: for each label row with a timestamp `t`, it looks up feature values that were valid *at time t* — preventing future data from leaking into the training set.

In [ ]:
# Cell 4: Defining Features with Feast FeatureView

try:
    from feast import (
        Entity, FeatureView, FeatureStore, Field,
        FileSource, RepoConfig,
    )
    from feast.types import Float32, Float64, Int64, String
    from feast.infra.online_stores.sqlite import SqliteOnlineStoreConfig
    FEAST_AVAILABLE = True
    print('Feast installed — using real Feast FeatureStore')
except ImportError:
    FEAST_AVAILABLE = False
    print('Feast not installed — running mock implementation (functionally equivalent)')

if FEAST_AVAILABLE:
    # ── Entity definition ──
    user = Entity(name='user', join_keys=['user_id'],
                  description='A registered e-commerce user')

    # ── Data Sources ──
    profile_source = FileSource(
        path=str(DATA_DIR / 'user_profiles.parquet'),
        timestamp_field='event_timestamp',
        created_timestamp_column='created',
    )
    stats_source = FileSource(
        path=str(DATA_DIR / 'user_stats.parquet'),
        timestamp_field='event_timestamp',
        created_timestamp_column='created',
    )

    # ── Feature Views ──
    user_profile_fv = FeatureView(
        name='user_profile',
        entities=[user],
        ttl=timedelta(days=30),
        schema=[
            Field(name='age',                  dtype=Int64),
            Field(name='account_age_days',     dtype=Int64),
            Field(name='country',              dtype=String),
            Field(name='subscription_tier',    dtype=String),
        ],
        source=profile_source,
        tags={'team': 'data-platform', 'criticality': 'high'},
    )

    user_stats_fv = FeatureView(
        name='user_stats',
        entities=[user],
        ttl=timedelta(hours=6),
        schema=[
            Field(name='purchases_last_30d',       dtype=Int64),
            Field(name='avg_order_value',          dtype=Float64),
            Field(name='session_count_last_7d',    dtype=Int64),
            Field(name='cart_abandonment_rate',    dtype=Float64),
            Field(name='days_since_last_purchase', dtype=Int64),
        ],
        source=stats_source,
        tags={'team': 'ml-platform', 'criticality': 'high'},
    )

    print('Feature views defined:')
    print(f'  user_profile: {len(user_profile_fv.schema)} features | TTL=30d')
    print(f'  user_stats:   {len(user_stats_fv.schema)} features | TTL=6h')

else:
    # ── Mock Feast-like implementation ──
    from dataclasses import dataclass, field as dc_field

    @dataclass
    class MockFeatureView:
        name: str
        entity_join_key: str
        feature_columns: list[str]
        ttl_hours: int
        source_path: str
        tags: dict = dc_field(default_factory=dict)

        def load(self) -> pd.DataFrame:
            return pd.read_parquet(self.source_path)

    user_profile_fv = MockFeatureView(
        'user_profile', 'user_id',
        ['age', 'account_age_days', 'country', 'subscription_tier'],
        ttl_hours=720, source_path=str(DATA_DIR / 'user_profiles.parquet'),
    )
    user_stats_fv = MockFeatureView(
        'user_stats', 'user_id',
        ['purchases_last_30d', 'avg_order_value', 'session_count_last_7d',
         'cart_abandonment_rate', 'days_since_last_purchase'],
        ttl_hours=6, source_path=str(DATA_DIR / 'user_stats.parquet'),
    )
    print('Mock FeatureViews created:')
    for fv in [user_profile_fv, user_stats_fv]:
        print(f'  {fv.name}: {fv.feature_columns}')

## Offline vs Online Store — Read and Write Patterns

### Offline Store
The offline store holds the **full history** of feature values timestamped over time. It is optimized for high-throughput sequential reads and is used to:
- Create **training datasets** with point-in-time joins
- Run **batch inference** over millions of entities
- Compute **aggregate statistics** for feature monitoring

Typical backend: Parquet files on S3/GCS, or Delta Lake / Iceberg tables for ACID compliance.

### Online Store
The online store holds only the **latest feature value per entity** (no history). It is optimized for low-latency point lookups (single entity or small batch) and is used exclusively for:
- **Real-time inference** — retrieving features for an incoming prediction request

Typical backend: Redis (most common), DynamoDB, Bigtable.

### Write Patterns
| Pattern | When | What | Destination |
|---|---|---|---|
| Batch write | Scheduled (hourly/daily) | All entities | Offline + Online |
| Stream write | On event arrival | Single entity | Online only |
| Historical backfill | One-time | All history | Offline only |

### Read Patterns
| Pattern | API | Source | Latency |
|---|---|---|---|
| Training dataset | `get_historical_features()` | Offline | Minutes |
| Batch scoring | `get_historical_features()` | Offline | Minutes |
| Real-time serving | `get_online_features()` | Online | <10 ms |

In [ ]:
# Cell 6: Writing to and Reading from the Feature Store

# ── 6a. Initialize the Feast FeatureStore (or Mock) ──
if FEAST_AVAILABLE:
    # Write feature_store.yaml
    fs_yaml = f"""
project: ecommerce_features
registry: {FEAST_REPO}/registry.pb
provider: local
online_store:
  type: sqlite
  path: {FEAST_REPO}/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 2
"""
    (FEAST_REPO / 'feature_store.yaml').write_text(fs_yaml.strip())

    # Write feature definitions module
    # (Feast requires a Python module in the repo dir at apply time)
    feature_def_code = '''
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float64, Int64, String
from datetime import timedelta
user = Entity(name="user", join_keys=["user_id"])
'''
    (FEAST_REPO / 'features.py').write_text(feature_def_code)

    store = FeatureStore(repo_path=str(FEAST_REPO))
    store.apply([user, user_profile_fv, user_stats_fv])
    print('Feast FeatureStore initialized and features applied.')

    # ── 6b. Materialize to online store ──
    end_dt   = datetime(2024, 12, 31, tzinfo=None)
    start_dt = end_dt - timedelta(days=1)
    try:
        store.materialize(start_date=start_dt, end_date=end_dt)
        print('Materialization complete: offline → online store synced.')
    except Exception as e:
        print(f'Materialization note: {e}')

    # ── 6c. Read from online store ──
    try:
        online_response = store.get_online_features(
            features=[
                'user_profile:age',
                'user_profile:subscription_tier',
                'user_stats:purchases_last_30d',
                'user_stats:avg_order_value',
            ],
            entity_rows=[{'user_id': uid} for uid in [1, 5, 100, 999]],
        ).to_df()
        print('Online feature retrieval:')
        print(online_response.to_string(index=False))
    except Exception as e:
        print(f'Online store note: {e}')

else:
    # ── Mock write/read simulation ──
    class MockOnlineStore:
        """Simulates Redis-like key-value online store (entity_id → feature dict)."""
        def __init__(self):
            self._store: dict[int, dict] = {}

        def write(self, df: pd.DataFrame, join_key: str, feature_cols: list[str]):
            # Keep only latest row per entity
            latest = df.sort_values('event_timestamp').groupby(join_key).last().reset_index()
            for _, row in latest.iterrows():
                self._store[row[join_key]] = {col: row[col] for col in feature_cols}
            print(f'Online store: {len(self._store)} entities loaded')

        def read(self, entity_ids: list[int], feature_cols: list[str]) -> pd.DataFrame:
            rows = []
            for eid in entity_ids:
                entry = self._store.get(eid, {})
                row = {'user_id': eid}
                row.update({col: entry.get(col, None) for col in feature_cols})
                rows.append(row)
            return pd.DataFrame(rows)

    online_store = MockOnlineStore()

    # Write profiles and stats to online store
    online_store.write(user_profiles, 'user_id', user_profile_fv.feature_columns)
    online_store.write(user_stats,    'user_id', user_stats_fv.feature_columns)

    # Read features for 4 users (simulating inference request)
    import time
    t0 = time.perf_counter()
    all_features = user_profile_fv.feature_columns + user_stats_fv.feature_columns
    result = online_store.read([1, 5, 100, 999], all_features)
    latency_ms = (time.perf_counter() - t0) * 1000

    print(f'Online read latency: {latency_ms:.2f}ms')
    print(result.to_string(index=False))

## Feature Serving in Production

### The Serving Request Flow
When a prediction request arrives at your ML service:

```
1. Request arrives:  POST /predict  {"user_id": 42, "item_id": 99}
2. Feature lookup:   online_store.get({user_id: 42, item_id: 99})
3. Feature join:     {age: 34, purchases_30d: 7, item_ctr: 0.08, ...}
4. Model inference:  model.predict(features) → 0.87
5. Response:         {"score": 0.87, "latency_ms": 8}
```

The full round-trip budget is typically under 100ms. Feature lookup must consume less than 10ms of that budget, which is why Redis (sub-millisecond P99) is the dominant online store choice.

### Production Considerations

**Freshness SLAs:** Define per-feature-view TTLs and alert when materialization jobs are late. A stale feature is often worse than a missing one.

**Missing value handling:** If an entity has never been seen (cold start), the online store returns `None`. Your serving code must handle this gracefully — either with a default value, a fallback model, or by rejecting the request with an explicit error.

**Batching requests:** For batch inference, never call `get_online_features()` in a for-loop over individual entities. The SDK supports bulk lookups — always send all entity keys in a single request.

**Feature logging:** Log every feature vector served at inference time. This enables:
- Training data collection for the next model version
- Drift detection (compare served distribution vs training distribution)
- Debugging predictions in production

In [ ]:
# Cell 8: Point-in-Time Correct Feature Retrieval and Training Dataset Creation

# ── 8a. Simulate a label dataset ──
# In practice this would come from your data warehouse:
# SELECT user_id, label, event_timestamp FROM conversions WHERE ...
label_df = pd.DataFrame({
    'user_id': np.random.choice(user_ids, 500, replace=False),
    'label':   np.random.randint(0, 2, 500),          # 1 = purchased, 0 = no purchase
    'event_timestamp': pd.to_datetime([
        datetime(2024, 1, 1) + timedelta(days=np.random.randint(0, 365))
        for _ in range(500)
    ]).tz_localize('UTC'),
})
print(f'Label dataset: {label_df.shape[0]} rows | positive rate: {label_df["label"].mean():.2%}')

# ── 8b. Point-in-time join (Feast or mock) ──
if FEAST_AVAILABLE:
    try:
        training_df = store.get_historical_features(
            entity_df=label_df,
            features=[
                'user_profile:age',
                'user_profile:account_age_days',
                'user_profile:subscription_tier',
                'user_stats:purchases_last_30d',
                'user_stats:avg_order_value',
                'user_stats:cart_abandonment_rate',
            ],
        ).to_df()
        print('Point-in-time join complete via Feast:')
        print(training_df.head(4).to_string(index=False))
    except Exception as e:
        print(f'Feast historical features note: {e}')
        training_df = None
else:
    def point_in_time_join(
        labels: pd.DataFrame,          # must have: entity_id, event_timestamp
        features: pd.DataFrame,        # must have: entity_id, event_timestamp, feature cols
        entity_col: str,
        feature_cols: list[str],
        ttl_hours: int = 6,
    ) -> pd.DataFrame:
        """For each label row, find the most recent feature row at or before label timestamp.
        Rows with no feature data within TTL window are dropped (data leakage prevention)."""
        labels = labels.copy()
        labels['event_timestamp'] = pd.to_datetime(labels['event_timestamp']).dt.tz_localize(None)
        features = features.copy()
        features['event_timestamp'] = pd.to_datetime(features['event_timestamp']).dt.tz_localize(None)

        merged_rows = []
        for _, label_row in labels.iterrows():
            uid  = label_row[entity_col]
            t    = label_row['event_timestamp']
            ttl  = timedelta(hours=ttl_hours)

            # Find latest feature row with timestamp <= label timestamp and within TTL
            user_feats = features[
                (features[entity_col] == uid) &
                (features['event_timestamp'] <= t) &
                (features['event_timestamp'] >= t - ttl)
            ]

            if user_feats.empty:
                continue  # no valid feature data — exclude this training row

            latest_feat = user_feats.sort_values('event_timestamp').iloc[-1]
            row = {entity_col: uid, 'label': label_row['label'], 'label_ts': t}
            row.update({col: latest_feat[col] for col in feature_cols})
            merged_rows.append(row)

        return pd.DataFrame(merged_rows)

    # Join profiles + stats at label time
    labels_tz_stripped = label_df.copy()
    labels_tz_stripped['event_timestamp'] = labels_tz_stripped['event_timestamp'].dt.tz_localize(None)

    training_df = point_in_time_join(
        labels_tz_stripped, user_stats, 'user_id',
        feature_cols=['purchases_last_30d', 'avg_order_value',
                      'session_count_last_7d', 'cart_abandonment_rate'],
        ttl_hours=6,
    )
    # Add static profile features via simple merge
    profile_feats = user_profiles[['user_id', 'age', 'account_age_days', 'subscription_tier']]
    training_df = training_df.merge(profile_feats, on='user_id', how='left')

    print(f'Training dataset after point-in-time join: {training_df.shape}')
    print(f'Rows retained (had valid features within TTL): {len(training_df)} / {len(label_df)}')
    print(training_df.head(4).to_string(index=False))

# ── 8c. Quick sanity-check model to validate the training dataset ──
if training_df is not None and len(training_df) > 100:
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import roc_auc_score

    feat_cols = ['age', 'account_age_days', 'purchases_last_30d',
                 'avg_order_value', 'session_count_last_7d', 'cart_abandonment_rate']
    feat_cols = [c for c in feat_cols if c in training_df.columns]

    X = training_df[feat_cols].fillna(0)
    y = training_df['label'].astype(int)

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = GradientBoostingClassifier(n_estimators=50, random_state=42)
    clf.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
    print(f'\nSanity-check model AUC-ROC: {auc:.4f} (expected ~0.5-0.7 for random labels)')

## Best Practices and Anti-Patterns

### Best Practices

**1. Define TTLs conservatively**
Set TTL to the maximum age at which a feature is still meaningful for the model. A `purchases_last_30d` feature with a 6-hour TTL ensures stale values are never served. If in doubt, shorter is safer.

**2. Log served features**
Every feature vector delivered to the model at inference time should be logged asynchronously to your offline store. This creates the ground truth for drift detection and enables future training on production distribution.

**3. Version feature views explicitly**
When changing a feature's computation logic (not just adding new features), create a new FeatureView (`user_stats_v2`) rather than mutating the existing one. This preserves backward compatibility for models that depend on the old definition.

**4. Monitor freshness, not just availability**
A health check that confirms Redis is running is not sufficient. Monitor the `max_event_timestamp` in your online store per FeatureView and alert if it falls more than 2× TTL behind real-time.

**5. Test with production data distributions**
Run your materialization pipeline against a sample of last week's production data before deploying new features. Null rates, distribution shifts, and cardinality explosions are far cheaper to catch pre-deployment.

---

### Anti-Patterns to Avoid

| Anti-Pattern | Why It's Dangerous | Correct Approach |
|---|---|---|
| Computing features inside the model server | Duplicates logic, causes skew, adds latency | Pre-compute and serve from feature store |
| Using `now()` as label timestamp | Leaks future data into training | Always use the actual event timestamp |
| One monolithic FeatureView | Hard to reuse, single point of failure | Split by entity and update frequency |
| Skipping the registry | Features undiscoverable, duplicated across teams | Register all features in a shared catalog |
| Silent null fallback | Model silently degrades on cold-start users | Explicit default values or model fallback with alerting |
| No feature monitoring | Distribution shifts go unnoticed for weeks | Track PSI / KS-test per feature on every materialization run |

In [ ]:
# Cell 10: Exercises
# Three hands-on challenges to cement your understanding of feature stores.

print('=' * 65)
print('EXERCISES')
print('=' * 65)
print('''
Challenge 1 — Incremental Materialization
-----------------------------------------
The MockOnlineStore.write() method currently reloads ALL entities
every time it is called. This is wasteful for large datasets.

Implement an `incremental_materialize(since: datetime)` function
that reads only rows with event_timestamp > since from the Parquet
file and updates only those entity entries in the online store.

Test it by:
  1. Writing a full snapshot at 2024-01-01
  2. Appending 100 new rows with timestamps in 2024-06-01
  3. Running incremental_materialize(since=datetime(2024-03-01))
  4. Confirming only the 100 new rows were written (check timestamps)

Why it matters: production feature stores materialize incrementally
to minimize compute cost and reduce latency of the sync window.
''')

print('''
Challenge 2 — Feature Freshness Monitor
----------------------------------------
Build a `check_feature_freshness(online_store, feature_views, alert_threshold_hours)'
function that:
  1. For each FeatureView, queries the online store for a sample of 100 entities
  2. Computes the max observed event_timestamp across those entities
  3. Compares it to datetime.now() and computes lag_hours
  4. If lag_hours > alert_threshold_hours, prints a WARNING with the FeatureView
     name, lag_hours, and TTL
  5. Returns a summary DataFrame: [feature_view, max_ts, lag_hours, status]

Then simulate a stale materialization by setting all event_timestamps in the
user_stats Parquet to 48 hours ago, re-running the check, and confirming
the alert fires correctly.
''')

print('''
Challenge 3 — On-Demand Feature Transformation
-----------------------------------------------
Feast supports "on-demand feature views" that compute derived features
at request time using request-time data + stored features.

Implement an OnDemandFeatureView class with a transform() method that:
  1. Accepts a dict of stored features (from get_online_features)
  2. Accepts a dict of request-time data (e.g., {'request_item_price': 49.99})
  3. Computes and returns derived features:
       - 'price_vs_avg_order_ratio'  = request_item_price / avg_order_value
       - 'high_value_user'           = 1 if avg_order_value > 100 else 0
       - 'days_since_purchase_bin'   = pd.cut(days_since_last_purchase, [0,7,30,90,999])

Then integrate it into a full serving pipeline:
  1. Fetch stored features for user_id=42 from online store
  2. Apply OnDemandFeatureView.transform() with item_price from the request
  3. Combine stored + derived features into a single prediction vector
  4. Time the full pipeline end-to-end (should be <50ms for a local Redis/dict)
''')